## NB3 - Modelling
### Goal
In this notebook, a genre-based movie recommendation system is developed.User can select their favourite genres and the system recommends movies by filtering relevant titles and ranking them using vote average and popularity
- load the movie data set
- prepare the genres column
- let user choose favourite genres
- filter matching movies
- rank them by rating and popularity
- return top recommendations


## (1) Import libraries

In [6]:
import pandas as pd
import ast

## (2) Load dataset

In [7]:
movies_df = pd.read_csv("C:/Projects/07_Movie_Recommendation_System/data/movies_dataset.csv")
movies_df.head()

,id,title,overview,genres,release_date,vote_average,vote_count,popularity,original_language
0,1265609,War Machine,On one last grueling mission during Army Range...,"['Action', 'Science Fiction', 'Thriller']",2026-02-12,7.256,1080,374.3784,en
1,875828,Peaky Blinders: The Immortal Man,After his estranged son gets embroiled in a Na...,"['Crime', 'Drama']",2026-03-05,7.400,298,370.3607,en
2,687163,Project Hail Mary,Science teacher Ryland Grace wakes up on a spa...,"['Science Fiction', 'Adventure']",2026-03-15,8.251,309,323.2718,en
3,1290821,Shelter,A man living in self-imposed exile on a remote...,"['Action', 'Crime', 'Thriller']",2026-01-28,6.600,359,323.2197,en
4,83533,Avatar: Fire and Ash,In the wake of the devastating war against the...,"['Science Fiction', 'Adventure', 'Fantasy']",2025-12-17,7.265,1913,319.3325,en


## (3) Basic Cleaning

In [8]:
# Convert genres 
movies_df["genres"] = movies_df["genres"].apply(ast.literal_eval)

In [9]:
# Remove rows with missing important fields
movies_df = movies_df.dropna(subset=["title", "genres", "vote_average", "popularity"])
movies_df.reset_index(drop=True)

,id,title,overview,genres,release_date,vote_average,vote_count,popularity,original_language
0,1265609,War Machine,On one last grueling mission during Army Range...,"[Action, Science Fiction, Thriller]",2026-02-12,7.256,1080,374.3784,en
1,875828,Peaky Blinders: The Immortal Man,After his estranged son gets embroiled in a Na...,"[Crime, Drama]",2026-03-05,7.400,298,370.3607,en
2,687163,Project Hail Mary,Science teacher Ryland Grace wakes up on a spa...,"[Science Fiction, Adventure]",2026-03-15,8.251,309,323.2718,en
3,1290821,Shelter,A man living in self-imposed exile on a remote...,"[Action, Crime, Thriller]",2026-01-28,6.600,359,323.2197,en
4,83533,Avatar: Fire and Ash,In the wake of the devastating war against the...,"[Science Fiction, Adventure, Fantasy]",2025-12-17,7.265,1913,319.3325,en
...,...,...,...,...,...,...,...,...,...
195,542559,Last Summer,"During a long hot summer in the 1970s, four bo...",[Drama],2018-06-07,6.688,16,21.6970,en
196,519182,Despicable Me 4,"Gru and Lucy and their girls—Margo, Edith and ...","[Family, Comedy, Animation, Science Fiction, A...",2024-06-20,6.988,3111,28.6912,en
197,434714,My Days of Mercy,The daughter of a man on death row falls in lo...,"[Romance, Drama]",2018-03-31,7.300,244,27.9448,en
198,102195,Way... Way Out,A platonically wed American couple run a lunar...,"[Comedy, Science Fiction]",1966-10-26,5.700,18,26.3031,en


## (4) Check available genres

In [11]:
all_genres = sorted(
    set(
        genre
        for genres_list in movies_df["genres"]
        for genre in genres_list
    )
)
all_genres

['Action',
 'Adventure',
 'Animation',
 'Comedy',
 'Crime',
 'Documentary',
 'Drama',
 'Family',
 'Fantasy',
 'History',
 'Horror',
 'Music',
 'Mystery',
 'Romance',
 'Science Fiction',
 'TV Movie',
 'Thriller',
 'War',
 'Western']

## (5) Build recommendation function

In [14]:
def recommend_by_genres(selected_genres, df=movies_df, top_n=10):
    # filter movies that match at least one selected genre
    filtered_df = df[df["genres"].apply(lambda genres: any(genre in genres for genre in selected_genres))].copy()
    
    # sort by rating first, then popularity
    filtered_df = filtered_df.sort_values(
        by=["vote_average", "popularity"],
        ascending=False
    )
    
    # remove duplicates by title if any
    filtered_df = filtered_df.drop_duplicates(subset=["title"])
    
    return filtered_df[["title", "genres", "vote_average", "popularity", "release_date"]].head(top_n)

## (6) Test the recommender

In [15]:
# for simple genre
recommend_by_genres(["Action"], top_n=5)

,title,genres,vote_average,popularity,release_date
138,The Dark Knight,"[Action, Crime, Thriller]",8.528,31.7384,2008-07-16
135,The Lord of the Rings: The Fellowship of the Ring,"[Adventure, Fantasy, Action]",8.429,31.0981,2001-12-18
167,Inception,"[Action, Science Fiction, Adventure]",8.370,28.9330,2010-07-15
139,Chainsaw Man - The Movie: Reze Arc,"[Animation, Action, Romance, Fantasy]",8.329,31.4742,2025-09-19
105,Avengers: Infinity War,"[Adventure, Action, Science Fiction]",8.234,44.9002,2018-04-25


In [16]:
# For romance and drama
recommend_by_genres(["Romance" , "Drama"], top_n=10)

,title,genres,vote_average,popularity,release_date
56,The Shawshank Redemption,"[Drama, Crime]",8.717,54.4910,1994-09-23
109,The Godfather,"[Drama, Crime]",8.687,40.8372,1972-03-14
182,Parasite,"[Comedy, Thriller, Drama]",8.493,26.5272,2019-05-30
73,Interstellar,"[Adventure, Drama, Science Fiction]",8.468,50.3228,2014-11-05
139,Chainsaw Man - The Movie: Reze Arc,"[Animation, Action, Romance, Fantasy]",8.329,31.4742,2025-09-19
43,David,"[Animation, Family, Drama]",8.100,72.7995,2025-12-14
170,Oppenheimer,"[Drama, History]",8.033,34.1419,2023-07-19
175,Stereotype,"[Drama, Comedy]",8.000,23.1799,1997-11-30
152,Titanic,"[Drama, Romance]",7.903,29.7551,1997-12-18
52,Forgotten Love,"[Drama, Romance]",7.800,41.4909,2023-09-26


In [17]:
# For Comedy + Family
recommend_by_genres(["Comedy" , "Family"], top_n=10)

,title,genres,vote_average,popularity,release_date
174,Spirited Away,"[Animation, Family, Fantasy]",8.534,27.4352,2001-07-20
182,Parasite,"[Comedy, Thriller, Drama]",8.493,26.5272,2019-05-30
186,Pulp Fiction,"[Thriller, Crime, Comedy]",8.486,26.6678,1994-09-10
43,David,"[Animation, Family, Drama]",8.100,72.7995,2025-12-14
66,KPop Demon Hunters,"[Fantasy, Music, Comedy, Animation]",8.040,56.2494,2025-06-20
175,Stereotype,"[Drama, Comedy]",8.000,23.1799,1997-11-30
145,Zootopia,"[Animation, Adventure, Family, Comedy]",7.760,29.7453,2016-02-11
9,Zootopia 2,"[Animation, Comedy, Adventure, Family, Mystery]",7.610,179.3031,2025-11-26
10,Hoppers,"[Animation, Family, Science Fiction, Comedy]",7.607,139.3869,2026-03-04
93,The Super Mario Bros. Movie,"[Family, Comedy, Adventure, Animation, Fantasy]",7.600,50.6437,2023-04-05


## (7) Stronger Version

In [18]:
def recommend_by_genres_v2(selected_genres, df=movies_df, top_n=10):
    temp_df = df.copy()

    # count how many selected genres match each movie
    temp_df["genre_match_count"] = temp_df["genres"].apply(
        lambda genres: sum(genre in genres for genre in selected_genres)

    )

    # keep omly movies with at least 1 match
    temp_df = temp_df[temp_df["genre_match_count"]>0]

    temp_df = temp_df.sort_values(
        by=["genre_match_count", "vote_average", "popularity"],
        ascending=False
    )

## (8) Test 

In [22]:
recommend_by_genres_v2(["Action", "Thriller"], top_n=10)
recommend_by_genres_v2

<function __main__.recommend_by_genres_v2(selected_genres, df=          id                                     title  \
0    1265609                               War Machine   
1     875828          Peaky Blinders: The Immortal Man   
2     687163                         Project Hail Mary   
3    1290821                                   Shelter   
4      83533                      Avatar: Fire and Ash   
..       ...                                       ...   
195   542559                               Last Summer   
196   519182                           Despicable Me 4   
197   434714                          My Days of Mercy   
198   102195                            Way... Way Out   
199      671  Harry Potter and the Philosopher's Stone   

                                              overview  \
0    On one last grueling mission during Army Range...   
1    After his estranged son gets embroiled in a Na...   
2    Science teacher Ryland Grace wakes up on a spa...   
3    A ma

## (9) Optional 

In [25]:
def recommend_strict_genres(selected_genres, df=movies_df, top_n=10):
    filtered_df = df[df["genres"].apply(lambda genres: all(genre in genres for genre in selected_genres))].copy()
    
    filtered_df = filtered_df.sort_values(
        by=["vote_average", "popularity"],
        ascending=False
    )
    
    filtered_df = filtered_df.drop_duplicates(subset=["title"])
    
    return filtered_df[["title", "genres", "vote_average", "popularity", "release_date"]].head(top_n)

In [26]:
# Test
recommend_strict_genres(["Action", "Thriller"], top_n=10)
recommend_strict_genres

<function __main__.recommend_strict_genres(selected_genres, df=          id                                     title  \
0    1265609                               War Machine   
1     875828          Peaky Blinders: The Immortal Man   
2     687163                         Project Hail Mary   
3    1290821                                   Shelter   
4      83533                      Avatar: Fire and Ash   
..       ...                                       ...   
195   542559                               Last Summer   
196   519182                           Despicable Me 4   
197   434714                          My Days of Mercy   
198   102195                            Way... Way Out   
199      671  Harry Potter and the Philosopher's Stone   

                                              overview  \
0    On one last grueling mission during Army Range...   
1    After his estranged son gets embroiled in a Na...   
2    Science teacher Ryland Grace wakes up on a spa...   
3    A m

## (10) Save cleaned dataset for app use

In [27]:
movies_df.to_csv("movies_dataset_cleaned.csv", index=False)
print("Cleaned dataset saved")

Cleaned dataset saved
